# 📓 Semana 2 · Dia 4 — Modelagem dimensional e Star Schema

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (modelagem), DEP (SCD) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Modelo conceitual do projeto documentado |

---


## 📖 Teoria — Por que modelar?

Dados não modelados = consultas lentas e números divergentes. O padrão da indústria para análise é a **modelagem dimensional** (Kimball): separar o que é **fato** (medidas, eventos que acontecem — vendas) do que é **dimensão** (entidades de contexto — cliente, produto, tempo, loja).


## 📖 Teoria — Fato vs Dimensão

**Tabela fato**: registra eventos/medidas. Alta cardinalidade (muitas linhas), colunas numéricas aditivas, chaves estrangeiras para dimensões.
**Tabela dimensão**: contexto descritivo. Poucas linhas, colunas textuais, chave primária (surrogate key).

**Star Schema**: 1 fato no centro + dimensões ao redor (forma de estrela). É o padrão de consumo para BI e para a camada **Ouro** da Medallion.

```
         dim_cliente
             │
dim_tempo ──fato_vendas── dim_produto
             │
         dim_loja
```


## 📖 Teoria — SCD — Slowly Changing Dimensions

Dimensões mudam lentamente (cliente muda de cidade). Como registrar a história?

- **SCD1**: sobrescreve o valor antigo (perde histórico). Simples; usado quando o histórico não importa.
- **SCD2**: mantém histórico com versões (linhas com `valid_from`, `valid_to`, `is_current`). Complexo; usado em auditoria e análise histórica.
- **SCD3**: guarda só o valor anterior em coluna separada (`cidade_atual`, `cidade_anterior`). Raramente usado.

> 🎯 **Dica de prova**: a DEA/DEP cobra **quando usar SCD1 vs SCD2**: correção de dados = SCD1; histórico obrigatório = SCD2; quantidade de versões pequena e fixa = SCD3. Na Semana 8 implementamos SCD2 real com `APPLY CHANGES INTO`.


### 💻 Na prática — Desenhando o modelo do projeto

Vamos definir o modelo conceitual do nosso varejo (vamos materializar na Semana 4).


In [ ]:
# Modelo conceitual documentado em células markdown (abaixo) e validado aqui
modelo = """
FATO:  fato_vendas (data, dim_cliente, dim_produto, dim_loja, qtd, valor)
DIMS:  dim_cliente, dim_produto, dim_tempo, dim_loja
SCD:   dim_cliente -> SCD2 (cidade muda; precisamos histórico)
       dim_produto -> SCD1 (correção de descrição)
"""
print(modelo)

## 📖 Teoria — Convenções de nomenclatura do projeto

Adotamos as convenções padrão do mercado (usadas no Databricks e em entrevistas):
- Tabelas de camada: `workspace.bronze.*`, `workspace.prata.*`, `workspace.ouro.*`
- Dimensões: `dim_*` · Fatos: `fato_*` · Agregados de negócio: `nome_do_kpi`
- Views: sufixo `_vw` · Tabelas com `_ingested_at` no Bronze
- Chaves: `sk_*` (surrogate key) nas dimensões, `*_id` nas chaves naturais


> 🎯 **Dica de prova**: A DEA cobra identificar qual tabela é fato vs dimensão e qual esquema é Star vs Snowflake (normalizado). Star = denormalizado, mais rápido para BI.


## 🎯 Exercícios de fixação

**1.** Classifique: `dim_cliente`, `fato_vendas`, `dim_produto`, `dim_tempo` — qual é fato?

**2.** Por que o Ouro costuma ser denormalizado (star schema) em vez de normalizado?

**3.** Se o endereço de um cliente muda e você precisa do histórico de endereços, qual SCD usar?

**4.** Desenhe o star schema do projeto com pelo menos 3 dimensões.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Fato

`fato_vendas` — registra o evento (venda) com medidas numéricas; as demais são dimensões de contexto.

**2.** Denormalizado

BI consulta por agregação — joins com dimensões pequenas e diretas são mais rápidos e simples para o usuário final. Snowflake (normalizado) economiza espaço, mas complexifica a consulta.

**3.** SCD2

SCD2 preserva o histórico completo de versões do endereço — necessário para auditoria e análise temporal.

**4.** Star schema

Desenhe `fato_vendas` no centro, ligado a `dim_cliente`, `dim_produto`, `dim_tempo`, `dim_loja`. Cada dim com chave surrogate (sk_) ligada ao fato.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*